# DAY 5: Rigorous Evaluation: Base Model vs. Fine-Tuned Model

## 🎯 Learning Objectives
Never settle for "it looks better". In production LLM engineering, you must prove improvement quantitatively.

By the end of this day, you will be able to execute and explain:
1. **Held-Out Test Benchmarking**: Evaluating the Base Model and Fine-Tuned Model against the exact same unseen test split.
2. **Structured Evaluation Metrics**:
   - **JSON Validity Rate**: Proportion of outputs that successfully parse into valid JSON.
   - **Schema Conformance Rate**: Presence and type-correctness of all 5 schema keys.
   - **Exact Match Rate**: Strict equality across all extracted fields simultaneously.
   - **Field-Level Accuracy**: Accuracy for `customer`, `quantity`, `product`, `amount`, `delivery_day`.
3. **Inference Latency & Token Efficiency**: Comparing response length and generation speed.
4. **Failure Mode Analysis**: Diagnosing why base models fail (hallucination, markdown wrapping, preamble chat) vs. how fine-tuning enforces strict formatting.

In [1]:
!pip install -q transformers peft accelerate datasets pyyaml

In [6]:
import os
from src.data.dataset_generator import generate_order_dataset, save_dataset_splits

# Generate the exact deterministic splits if not already present
if not os.path.exists("data/processed/test.jsonl"):
    train_set, val_set, test_set = generate_order_dataset(
        total_train=1200,
        total_val=150,
        total_test=150,
        seed=42
    )
    save_dataset_splits(train_set, val_set, test_set, output_dir="data/processed")
    print("✅ Deterministic splits generated and saved to data/processed/")


Saved  1200 samples to data/processed/train.jsonl
Saved   150 samples to data/processed/val.jsonl
Saved   150 samples to data/processed/test.jsonl
✅ Deterministic splits generated and saved to data/processed/


In [3]:
import os
import sys

# 1. Clone or update repository on Colab
if not os.path.exists("/content/Fine-Tuning-PoCs"):
    !git clone https://github.com/sagarsambhwani/Fine-Tuning-PoCs.git
else:
    %cd /content/Fine-Tuning-PoCs
    !git pull origin main

# 2. Change directory to project root
%cd /content/Fine-Tuning-PoCs/llm-finetuning-production

# 3. Add to Python path so 'src' is discoverable
project_root = os.path.abspath("/content/Fine-Tuning-PoCs/llm-finetuning-production")
if project_root not in sys.path:
    sys.path.append(project_root)


Cloning into 'Fine-Tuning-PoCs'...
remote: Enumerating objects: 319, done.
remote: Counting objects: 100% (319/319), done.
remote: Compressing objects: 100% (189/189), done.
remote: Total 319 (delta 132), reused 305 (delta 118), pack-reused 0 (from 0)
Receiving objects: 100% (319/319), 204.20 KiB | 2.49 MiB/s, done.
Resolving deltas: 100% (132/132), done.
/content/Fine-Tuning-PoCs/llm-finetuning-production


In [4]:
import json
import time
import torch
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

from src.evaluation.metrics import evaluate_predictions
from src.data.formatter import format_chat_prompt, apply_template_to_messages

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER_PATH = "models/adapters/qwen-1.5b-order-extractor"
TEST_FILE = "data/processed/test.jsonl"

device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

print(f"Device: {device} ({torch_dtype})")

Device: cuda (torch.float16)


### Step 1: Load Held-Out Test Set

In [7]:
test_samples = []
with open(TEST_FILE, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            test_samples.append(json.loads(line))

print(f"Loaded {len(test_samples)} held-out test samples from {TEST_FILE}.")
print(f"First test input: {test_samples[0]['input']}")
print(f"First target: {test_samples[0]['output']}")

Loaded 150 held-out test samples from data/processed/test.jsonl.
First test input: Procurement request from Noah: 3 noise-canceling headphones worth $1077, please ensure arrival on Sunday.
First target: {'customer': 'Noah', 'quantity': 3, 'product': 'noise-canceling headphones', 'amount': 1077.0, 'delivery_day': 'Sunday'}


### Step 2: Benchmark Base Model (Zero-Shot)

In [8]:
print(f"Loading Base Model: {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch_dtype,
    device_map="auto" if torch.cuda.is_available() else None
)
base_model.eval()

base_predictions = []
ground_truths = [s["output"] for s in test_samples]

start_t = time.time()
for idx, sample in enumerate(test_samples):
    msgs = format_chat_prompt(sample["input"])
    prompt_text = apply_template_to_messages(msgs, tokenizer, add_generation_prompt=True)
    inp = tokenizer(prompt_text, return_tensors="pt").to(device)
    
    with torch.no_grad():
        out = base_model.generate(**inp, max_new_tokens=100, do_sample=False)
    
    gen_text = tokenizer.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    base_predictions.append(gen_text)

base_time = time.time() - start_t
base_metrics, base_details = evaluate_predictions(base_predictions, ground_truths)
print("=== BASE MODEL METRICS ===")
print(f"JSON Validity Rate : {base_metrics['json_validity_rate'] * 100:.2f}%")
print(f"Exact Match Rate   : {base_metrics['exact_match_rate'] * 100:.2f}%")
print(f"Field Accuracies   : {base_metrics['field_accuracies']}")

Loading Base Model: Qwen/Qwen2.5-1.5B-Instruct...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

=== BASE MODEL METRICS ===
JSON Validity Rate : 100.00%
Exact Match Rate   : 87.33%
Field Accuracies   : {'quantity': 0.9933, 'customer': 0.94, 'product': 0.98, 'amount': 0.9533, 'delivery_day': 1.0}


### Step 3: Benchmark Fine-Tuned Model (with Adapter)

In [12]:
%cd /content/Fine-Tuning-PoCs
!git pull origin main
%cd /content/Fine-Tuning-PoCs/llm-finetuning-production


/content/Fine-Tuning-PoCs
remote: Enumerating objects: 27, done.
remote: Counting objects: 100% (27/27), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 25 (delta 7), reused 25 (delta 7), pack-reused 0 (from 0)
Unpacking objects: 100% (25/25), 105.50 MiB | 10.86 MiB/s, done.
From https://github.com/sagarsambhwani/Fine-Tuning-PoCs
 * branch            main       -> FETCH_HEAD
   5de7e80..9b405e6  main       -> origin/main
Updating 5de7e80..9b405e6
Updating files: 100% (28/28), done.
Fast-forward
 .../adapters/qwen-1.5b-order-extractor/README.md   |     62 +
 .../qwen-1.5b-order-extractor/adapter_config.json  |     50 +
 .../adapter_model.safetensors                      |    Bin 0 -> 36981856 bytes
 .../qwen-1.5b-order-extractor/chat_template.jinja  |     54 +
 .../checkpoint-400/README.md                       |    209 +
 .../checkpoint-400/adapter_config.json             |     50 +
 .../checkpoint-400/adapter_model.safetensors       |    Bin 0 -> 36981856 bytes
 .

In [14]:
!pip install -U torchao


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 57.3 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [15]:
print(f"Attaching Fine-Tuned Adapter from {ADAPTER_PATH}...")
ft_model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
ft_model.eval()


Attaching Fine-Tuned Adapter from models/adapters/qwen-1.5b-order-extractor...


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(

In [16]:
print(f"Attaching Fine-Tuned Adapter from {ADAPTER_PATH}...")
ft_model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
ft_model.eval()

ft_predictions = []
start_t = time.time()
for idx, sample in enumerate(test_samples):
    msgs = format_chat_prompt(sample["input"])
    prompt_text = apply_template_to_messages(msgs, tokenizer, add_generation_prompt=True)
    inp = tokenizer(prompt_text, return_tensors="pt").to(device)
    
    with torch.no_grad():
        out = ft_model.generate(**inp, max_new_tokens=100, do_sample=False)
    
    gen_text = tokenizer.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    ft_predictions.append(gen_text)

ft_time = time.time() - start_t
ft_metrics, ft_details = evaluate_predictions(ft_predictions, ground_truths)
print("=== FINE-TUNED MODEL METRICS ===")
print(f"JSON Validity Rate : {ft_metrics['json_validity_rate'] * 100:.2f}%")
print(f"Exact Match Rate   : {ft_metrics['exact_match_rate'] * 100:.2f}%")
print(f"Field Accuracies   : {ft_metrics['field_accuracies']}")

Attaching Fine-Tuned Adapter from models/adapters/qwen-1.5b-order-extractor...


/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:305: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


=== FINE-TUNED MODEL METRICS ===
JSON Validity Rate : 100.00%
Exact Match Rate   : 100.00%
Field Accuracies   : {'quantity': 1.0, 'customer': 1.0, 'product': 1.0, 'amount': 1.0, 'delivery_day': 1.0}


### Step 4: Comparison Table & Report Generation

In [17]:
print("=" * 70)
print(f"{'Metric':<28} | {'Base Model':<16} | {'Fine-Tuned Model':<16}")
print("=" * 70)
print(f"{'JSON Validity Rate':<28} | {base_metrics['json_validity_rate']*100:>14.2f}% | {ft_metrics['json_validity_rate']*100:>14.2f}%")
print(f"{'Exact Match Rate':<28} | {base_metrics['exact_match_rate']*100:>14.2f}% | {ft_metrics['exact_match_rate']*100:>14.2f}%")
print(f"{'Avg Field Accuracy':<28} | {base_metrics['avg_field_accuracy']*100:>14.2f}% | {ft_metrics['avg_field_accuracy']*100:>14.2f}%")
for k in base_metrics['field_accuracies']:
    print(f"  - {k:<24} | {base_metrics['field_accuracies'][k]*100:>14.2f}% | {ft_metrics['field_accuracies'][k]*100:>14.2f}%")
print("=" * 70)

Metric                       | Base Model       | Fine-Tuned Model
JSON Validity Rate           |         100.00% |         100.00%
Exact Match Rate             |          87.33% |         100.00%
Avg Field Accuracy           |          97.33% |         100.00%
  - quantity                 |          99.33% |         100.00%
  - customer                 |          94.00% |         100.00%
  - product                  |          98.00% |         100.00%
  - amount                   |          95.33% |         100.00%
  - delivery_day             |         100.00% |         100.00%


---
## 🎓 Day 5 Interview Preparation & Concept Mastery

### 🧠 Concepts You Must Know
1. **Exact Match vs Field-Level Precision/Recall**: In structured tasks, exact match is the strictest metric (all fields correct). Field-level accuracy reveals granular entity extraction performance.
2. **Greedy Decoding for Evaluation**: Always use deterministic greedy decoding (`temperature=0.0`, `do_sample=False`) during benchmark evaluations to ensure reproducibility.
3. **Preamble Hallucination Elimination**: Fine-tuning teaches the model to immediately output `{` and close with `}`, omitting conversational preambles that break downstream JSON parsers.
4. **LLM-as-a-Judge**: Using a larger model (e.g. GPT-4) to evaluate free-form text. For structured JSON with known ground truth, rule-based schema validators are strictly superior and deterministic.

### 💼 5 Interview Questions
1. **Q1 (Beginner)**: *Why is Perplexity or Training Loss insufficient to prove your fine-tuned model works in production?*
   - **Answer**: Perplexity only measures next-token probability prediction. A model can have low perplexity while still outputting malformed JSON or omitting required fields. Downstream task metrics (JSON validity, Schema Conformance, Field Accuracy) are the true test of functional utility.
2. **Q2 (Intermediate)**: *Why is temperature set to 0.0 during evaluation?*
   - **Answer**: Setting temperature to 0.0 forces the model to select the argmax (highest probability) token at each step, removing sampling variance and ensuring exact reproducibility across benchmark runs.
3. **Q3 (Intermediate)**: *What was the most common failure mode of the base model before fine-tuning?*
   - **Answer**: Chattiness (e.g., adding "Here is your JSON:" or markdown backticks ```json) and occasional field naming variations (e.g., "total" instead of "amount").
4. **Q4 (Advanced)**: *How do you evaluate structured JSON extraction when field values can have synonyms (e.g. "laptops" vs "laptop computers")?*
   - **Answer**: By applying canonicalization functions during evaluation (e.g. lowercasing, stripping punctuation, or mapping synonyms to standard ontology dictionaries) or evaluating string semantic similarity via embeddings.
5. **Q5 (Advanced)**: *How would you set up automated continuous evaluation (CI/CD) for model fine-tuning?*
   - **Answer**: Run an automated evaluation pipeline in GitHub Actions or model registry webhooks that runs the model on the gold test set upon PR, blocking deployment if Exact Match Rate or JSON Validity falls below established SLA thresholds.

### 🛠️ 3 Practical Questions on Your Code
1. *In your code, how did you calculate field-level accuracy across the 5 schema keys?*
2. *How did `evaluate_single_sample()` handle cases where the raw model string was not valid JSON?*
3. *What was the exact percentage improvement in Exact Match Rate achieved by fine-tuning?*

### 🏆 Day 5 Hands-On Challenge
> **Challenge**: Implement an error breakdown function that groups failed test samples by error category: `SYNTAX_ERROR`, `MISSING_FIELD`, `VALUE_MISMATCH`, or `TYPE_ERROR`.